# Predictability ≠ Profitability — Narrative Walkthrough

A guided tour of the research arc: from *"can we predict BTC?"* to a
rigorously validated **no**, with one genuinely predictable target
(volatility) that turns out to be **trivial** and **unprofitable**.

Each section reproduces a real result. Full studies live behind the CLI
(`python main.py ...`); here we show the punchlines.

## 0. Setup
We load candles from the local SQLite store (collected from Binance).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import config
from src.data.database import Database

db = Database()
candles = db.load_candles(limit=50_000)
print(f'{len(candles):,} candles loaded')
candles[['open_time','open','high','low','close','volume']].tail(3)

## 1. The cost wall
Before any model: how far does price move vs. what it costs to trade?
This single comparison foreshadows every negative result.

In [ ]:
close = candles['close']
cost = 2*(config.BACKTEST['fee_pct'] + config.BACKTEST['slippage_pct'])
for H in (5, 15, 60):
    move = (close.shift(-H)/close - 1).abs().median()*100
    flag = 'move < cost' if move < cost else 'move > cost'
    print(f'H={H:>3} min   median |move| = {move:5.3f}%   ({flag})')
print(f'\nround-trip cost = {cost:.3f}%')

**Takeaway.** At 5 min the typical move (~0.05%) is *less than half* the
round-trip cost (~0.12%). Direction prediction is fighting uphill before
the model even starts.

## 2. Direction is noise
Across 8k / 100k / 250k candles and every horizon, directional AUC ≈ 0.50.
The one statistically significant edge (51.2% at 5 min, p=0.002) loses
~150% after costs. See `reports/horizon_study_report.md`.

In [ ]:
from IPython.display import Image
Image(filename='../reports/horizon_accuracy_across_sizes.png')

## 3. What *is* predictable? Compare targets on one ruler (AUC)
Same XGBoost, same features — only the **target** changes. Volatility
towers over direction.

In [ ]:
Image(filename='../reports/target_predictability.png')

## 4. The humbling: purged CV vs. a one-line baseline
Volatility's AUC ≈ 0.79 *looks* like an ML win. We race it against a
trivial persistence baseline (`vol_{t+1} ≈ vol_t`) under **purged,
embargoed** cross-validation. Watch what happens.

In [ ]:
from src.validation.vol_validation import run_purged_vol_cv
cv = run_purged_vol_cv(db.load_candles(limit=40_000), horizon=15, n_splits=4)
print(f"XGBoost (model)      AUC = {cv['model_auc_mean']:.3f} ± {cv['model_auc_std']:.3f}")
print(f"persistence baseline AUC = {cv['baseline_auc_mean']:.3f} ± {cv['baseline_auc_std']:.3f}")
print(f"model gain over baseline = {cv['model_beats_baseline']:+.3f}")

**Takeaway.** The trivial baseline **beats** the model. The celebrated
predictability was volatility *clustering* — a textbook stylized fact — and
the XGBoost with TA features is a *worse* way to capture it than one line
of code. Always race your model against the dumbest predictor.

## 5. Predictability ≠ profitability
Using volatility as a *filter* (trade only high-movement regimes) improves
profit factor by removing cost-dominated trades — but no strategy turns
profitable. The **Deflated Sharpe Ratio across 12 configurations is 0.00**.
See `reports/vol_economics_report.md` and `reports/validation_report.md`.

In [ ]:
Image(filename='../reports/vol_econ_etapa1.png')

## 6. Conclusion

| Question | Answer |
|---|---|
| Is direction predictable after costs? | **No** (AUC ≈ 0.50) |
| Does horizon fix it? | **No** |
| Is anything predictable? | **Volatility** (AUC ≈ 0.79)… |
| …as a model achievement? | **No** — a persistence baseline wins under purged CV |
| Any profitable strategy? | **No** — Deflated Sharpe ≈ 0.00 |

The value of this project is the **method**: falsification, out-of-sample
discipline, cost realism, autocorrelation-aware significance, purged CV, and
selection-bias correction — and the honesty to report a negative result.